# CNN Car Image Classifier — 7 Classes

**PPR501 — Machine Learning for Engineers (Applied Machine Learning)**

The problem addressed in this part is **car-image recognition** using a convolutional
neural network (CNN), with the output being one of the seven brand groups listed below.

The architecture used is **MobileNetV2** — a relatively compact CNN that was
pre-trained on a very large image dataset. As a result, a short fine-tuning stage is
sufficient to obtain good performance, even though this project's dataset contains
only a few thousand images.

| # | Class |
|---|-------|
| 0 | Audi |
| 1 | Hyundai Creta |
| 2 | Mahindra Scorpio |
| 3 | Rolls Royce |
| 4 | Maruti Suzuki Swift |
| 5 | Tata Safari |
| 6 | Toyota Innova |

## How to run

The notebook is **fully self-contained** — it does not depend on any other file in the
project; the entire source code lives in this notebook and the cells can be executed
sequentially.

**Libraries required:** TensorFlow, Keras, NumPy, Matplotlib, Scikit-learn, Gradio,
Pillow, Requests.

```bash
pip install tensorflow numpy matplotlib scikit-learn gradio pillow requests
```

## The dataset

The images come from the public Kaggle dataset used by the notebook
[Car Images Classification using CNN](https://www.kaggle.com/code/kshitij192/car-images-classification-using-cnn/data)
— dataset `kshitij192/cars-image-dataset`
(7 car brands, **3,352** train images and **813** test images, ≈ 38 MB).

The notebook takes care of everything by itself:

1. If a local copy is found (`Dataset/Train/<Class>/*.jpg` and `Dataset/Test/<Class>/*.jpg`
   next to the notebook or in the project tree), it is used directly.
2. Otherwise the dataset is **downloaded automatically from Kaggle** and extracted into
   `Dataset/Train/...` and `Dataset/Test/...` beside the notebook. No Kaggle account or
   API token is required — the download uses the public dataset endpoint.

So a user only needs this one ipynb file to run the whole project.

> ⏱️ The first run downloads ≈ 38 MB of images; the full training (20 epochs) takes
> roughly **10–15 minutes** on a typical CPU.

In [ ]:
# ============================================================
# CELL 0 - Install dependencies
# ============================================================
%pip install tensorflow numpy matplotlib scikit-learn gradio pillow requests

In [ ]:
# ============================================================
# CELL 1 - Imports
# ============================================================
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from PIL import Image

print(f"TensorFlow {tf.__version__}")

## 1. Configuration

All hyper-parameters are collected in a single location, which makes them convenient
to fine-tune later.

In [ ]:
# ============================================================
# CELL 2 - Configuration
# ============================================================
SEED = 42
IMAGE_SIZE = 224          # MobileNetV2 default input size
BATCH_SIZE = 32
NUM_CLASSES = 7
VALIDATION_SPLIT = 0.2   # 20% of Train is held out for validation

EPOCHS_FROZEN = 5        # Stage 1: frozen backbone
EPOCHS_FINETUNE = 15     # Stage 2: unfrozen backbone
LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
PATIENCE = 6

CLASS_NAMES = [
    "Audi",
    "Hyundai Creta",
    "Mahindra Scorpio",
    "Rolls Royce",
    "Maruti Suzuki Swift",
    "Tata Safari",
    "Toyota Innova",
]

tf.keras.utils.set_random_seed(SEED)

## 2. Downloading & locating the dataset

The dataset is **downloaded automatically from Kaggle** if it is not already present
locally, so that this notebook can be executed on its own — without any additional
file from the project.

* **Source**: [Car Images Classification using CNN](https://www.kaggle.com/code/kshitij192/car-images-classification-using-cnn/data)
  → Kaggle dataset [`kshitij192/cars-image-dataset`](https://www.kaggle.com/datasets/kshitij192/cars-image-dataset)
  (GPL 2.0, ≈ 38 MB, 3,352 train + 813 test images).
* **Local layout used by the notebook**:
  `Dataset/Train/<Class>/*.jpg` and `Dataset/Test/<Class>/*.jpg`.
* The download uses the **public** Kaggle dataset endpoint, so no API token is needed.
  If your environment blocks the download, place the images manually in the folders
  shown above and re-run this cell — the local copy is always preferred.

In [ ]:
# ============================================================
# CELL 3 - Download (if needed) and locate the dataset
# ============================================================
import shutil
import ssl
import urllib.request
import zipfile

# ---------------------------------------------------------------------------
# Dataset source — the public Kaggle dataset behind
# https://www.kaggle.com/code/kshitij192/car-images-classification-using-cnn/data
# ---------------------------------------------------------------------------
KAGGLE_DATASET = "kshitij192/cars-image-dataset"
KAGGLE_API_URL = f"https://www.kaggle.com/api/v1/datasets/download/{KAGGLE_DATASET}"
# The endpoint rejects requests without a Kaggle-like User-Agent.
KAGGLE_USER_AGENT = "kaggle-api-v1.6.17"

# Folder names used inside the dataset (the notebook labels them as shown below).
EXPECTED_FOLDERS = [
    "Audi",
    "Hyundai Creta",
    "Mahindra Scorpio",
    "Rolls Royce",
    "Swift",
    "Tata Safari",
    "Toyota Innova",
]


def _download_file(url: str, dest: Path) -> None:
    """Download `url` to `dest` (Requests first, urllib as a fallback)."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        import requests  # noqa: WPS433 (optional dependency)

        response = requests.get(
            url,
            stream=True,
            timeout=300,
            headers={"User-Agent": KAGGLE_USER_AGENT},
        )
        response.raise_for_status()
        with open(dest, "wb") as handle:
            for chunk in response.iter_content(chunk_size=1 << 20):
                if chunk:
                    handle.write(chunk)
        return
    except ImportError:
        pass  # Requests is not installed -> fall back to urllib

    context = None
    try:
        import certifi  # noqa: WPS433 (optional dependency)

        context = ssl.create_default_context(cafile=certifi.where())
    except Exception:
        context = ssl.create_default_context()

    request = urllib.request.Request(url, headers={"User-Agent": KAGGLE_USER_AGENT})
    with urllib.request.urlopen(request, context=context, timeout=300) as response:
        with open(dest, "wb") as handle:
            shutil.copyfileobj(response, handle)


def _extract_kaggle_archive(zip_path: Path, target_root: Path) -> int:
    """
    Extract the Kaggle archive into `target_root`.

    The archive stores the images as  <top folder>/train/<Class>/<file>.jpg
    and                            <top folder>/test/<Class>/<file>.jpg
    which is rewritten to          <target_root>/Train/<Class>/<file>.jpg
                                   <target_root>/Test/<Class>/<file>.jpg
    """
    extracted = 0
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.namelist():
            parts = member.split("/")
            if len(parts) < 3 or member.endswith("/"):
                continue
            split = parts[1].lower()
            if split not in ("train", "test"):
                continue
            relative = Path(*parts[2:])
            destination = target_root / split.capitalize() / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, open(destination, "wb") as handle:
                shutil.copyfileobj(source, handle)
            extracted += 1
    return extracted


def download_dataset(target_root: Path) -> Path:
    """Download the Kaggle dataset and extract it into `target_root`."""
    archive_path = target_root / "_cars_image_dataset.zip"

    print(f"Downloading dataset '{KAGGLE_DATASET}' from Kaggle ...")
    _download_file(KAGGLE_API_URL, archive_path)
    size_mb = archive_path.stat().st_size / 1e6
    print(f"Downloaded {size_mb:.1f} MB -> {archive_path}")

    extracted = _extract_kaggle_archive(archive_path, target_root)
    print(f"Extracted {extracted} images into {target_root}")
    archive_path.unlink(missing_ok=True)  # keep the workspace clean
    return target_root


def locate_dataset():
    """
    Return the folder that contains  Train/<Class>/*.jpg  and  Test/<Class>/*.jpg.

    A local copy is always preferred; otherwise the dataset is downloaded from
    Kaggle into  <notebook folder>/Dataset  and used from there.
    """
    here = Path.cwd()
    candidates = [
        here / "Dataset",
        here / "sources" / "Dataset",
        here.parent / "Dataset",
        here.parent / "sources" / "Dataset",
        here.parent.parent / "Dataset",
        here.parent.parent / "sources" / "Dataset",
    ]
    for candidate in candidates:
        if (candidate / "Train").is_dir() and (candidate / "Test").is_dir():
            return candidate.resolve()

    # Nothing found locally -> download the dataset from Kaggle.
    target_root = here / "Dataset"
    download_dataset(target_root)

    if not ((target_root / "Train").is_dir() and (target_root / "Test").is_dir()):
        raise FileNotFoundError(
            "The dataset could not be prepared.\n"
            "Expected layout:  <...>/Dataset/Train/<Class>/*.jpg  and  <...>/Dataset/Test/<Class>/*.jpg\n"
            "Please download it manually from "
            "https://www.kaggle.com/datasets/kshitij192/cars-image-dataset "
            "and place it in the folders shown above."
        )
    return target_root.resolve()


DATASET_DIR = locate_dataset()
TRAIN_DIR = DATASET_DIR / "Train"
TEST_DIR = DATASET_DIR / "Test"

print(f"Dataset : {DATASET_DIR}")
print(f"Train   : {TRAIN_DIR}")
print(f"Test    : {TEST_DIR}")

### Dataset check

Confirm that all seven classes are present and survey the number of images in each
class. These counts determine how the class imbalance is handled in the sections that
follow.

> Note: the class folders are named `Audi`, `Hyundai Creta`, `Mahindra Scorpio`,
> `Rolls Royce`, `Swift`, `Tata Safari`, `Toyota Innova`. The label
> **`Maruti Suzuki Swift`** used in the reports corresponds to the `Swift` folder.

In [ ]:
# ============================================================
# CELL 4 - Dataset overview
# ============================================================
def class_counts(dataset_dir: Path) -> Counter:
    """Count how many images each class folder contains."""
    counter = Counter()
    for folder in sorted(p for p in dataset_dir.iterdir() if p.is_dir()):
        counter[folder.name] = sum(1 for f in folder.iterdir() if f.is_file())
    return counter


train_counts = class_counts(TRAIN_DIR)
test_counts = class_counts(TEST_DIR)

print(f"{'Class folder':<22}{'Train':>8}{'Test':>8}")
print("-" * 38)
for name in sorted(train_counts):
    print(f"{name:<22}{train_counts[name]:>8}{test_counts.get(name, 0):>8}")
print("-" * 38)
print(f"{'TOTAL':<22}{sum(train_counts.values()):>8}{sum(test_counts.values()):>8}")

print("\nTrain classes:", sorted(train_counts))
print("Test  classes:", sorted(test_counts))

if sorted(train_counts) != sorted(test_counts):
    print("\n⚠️  The Train and Test folder names do NOT match - check the dataset.")

missing = [name for name in EXPECTED_FOLDERS if name not in train_counts]
if missing:
    print(f"\n⚠️  Missing expected class folders: {missing}")

# ---------------------------------------------------------------------------
# The label order used by Keras is the alphabetical order of the class folders:
#   Audi, Hyundai Creta, Mahindra Scorpio, Rolls Royce, Swift, Tata Safari, Toyota Innova
# The reports use the full brand name "Maruti Suzuki Swift" for the "Swift" folder,
# which is exactly the order declared in the configuration cell.
# ---------------------------------------------------------------------------
FOLDER_TO_DISPLAY = {
    "Swift": "Maruti Suzuki Swift",
}

folder_order = sorted(train_counts)
display_order = [FOLDER_TO_DISPLAY.get(name, name) for name in folder_order]

print("\nLabel order used by the model:")
for index, name in enumerate(display_order):
    print(f"  {index}: {name}")

if display_order != CLASS_NAMES:
    print("\n⚠️  The folder order does NOT match CLASS_NAMES from the configuration cell.")
    print(f"    config  : {CLASS_NAMES}")
    print(f"    from dir: {display_order}")
else:
    print("\n✅  The folder order matches CLASS_NAMES from the configuration cell.")

### On the class imbalance

In the training set, Audi contains 814 images while Hyundai Creta contains only 271 —
a gap of roughly 3×. If this is left untreated, the model becomes clearly biased
towards the majority class and predicts "Audi" for nearly every input image.

Two tools are applied together (details in the section immediately below and in section 7):

1. **Class weights** during training — rare classes are assigned a higher weight in the
   loss function.
2. **Prior correction** at inference — the output probabilities are re-adjusted
   according to the actual frequency of each class.

In [ ]:
# ============================================================
# CELL 5 - Class weights (effective number)
# ============================================================
def compute_class_weight(counts: Counter, mode: str = "effective") -> dict:
    """
    Compute the per-class loss weight.

    'effective' uses the effective-number formula of Cui et al. (2019):
        w_c = (1 - beta) / (1 - beta ** n_c)
    It is far gentler than plain inverse frequency, which suits transfer learning.

    Every weight is rescaled so the mean is exactly 1.0; without this the gradients
    are shrunk roughly 500x and the model learns almost nothing in the epoch budget.
    """
    folder_order = sorted(counts)
    total = sum(counts.values())
    n_classes = len(counts)

    if mode == "none":
        return {i: 1.0 for i in range(n_classes)}

    if mode == "balanced":
        raw = {i: total / (n_classes * counts[name]) for i, name in enumerate(folder_order)}
    else:  # effective
        beta = 0.999
        raw = {}
        for i, name in enumerate(folder_order):
            n = counts[name]
            raw[i] = (1.0 - beta) / (1.0 - beta**n) if n > 0 else 1.0

    mean = sum(raw.values()) / len(raw)
    return {i: w / mean for i, w in raw.items()}


class_weight = compute_class_weight(train_counts, mode="effective")

print("Effective-number class weights (mean = 1.0):")
for i, name in enumerate(sorted(train_counts)):
    print(f"  {name:<22} n={train_counts[name]:<5} w={class_weight[i]:.2f}")

print("Priors (relative frequency of each class):")
total_train = sum(train_counts.values())
PRIORS = np.array([train_counts[n] / total_train for n in sorted(train_counts)])
for name, p in zip(sorted(train_counts), PRIORS):
    print(f"  {name:<22} p={p:.3f}")

## 3. Data pipeline

The most important aspect of this section is **maintaining a strict separation between
the validation split and the test set**:

- `Dataset/Train` → 80% train + 20% validation
- `Dataset/Test` → used only for the final evaluation, **never** for model selection

Using the test set as a validation set (a common mistake) makes the reported accuracy
unreliable: the model has already "seen" those exact images during training.

In [ ]:
# ============================================================
# CELL 6 - Build tf.data datasets
# ============================================================
def build_datasets(image_size, batch_size, validation_split):
    """
    Build the train / validation / test datasets.

    Dataset/Train is split into train (80%) and validation (20%).
    Dataset/Test is used only for the final evaluation.
    Augmentation is applied to the training set only.
    """
    common = dict(
        image_size=(image_size, image_size),
        batch_size=batch_size,
        labels="inferred",
    )

    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, subset="training", seed=SEED, shuffle=True,
        validation_split=validation_split, **common
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, subset="validation", seed=SEED, shuffle=True,
        validation_split=validation_split, **common
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, shuffle=False, **common
    )

    # Normalise to [0, 1] for ALL splits.
    normalize = tf.keras.Sequential(
        [tf.keras.layers.Rescaling(1.0 / 255.0)], name="normalize"
    )

    # Mild augmentation, training only. Cars are roughly symmetric so a horizontal
    # flip is valid, but strong rotation/zoom would crop out the very details
    augment = tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.10),
            tf.keras.layers.RandomZoom(height_factor=0.10, width_factor=0.10),
            tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
            tf.keras.layers.RandomBrightness(factor=0.10),
        ],
        name="augmentation",
    )

    train_ds = train_ds.map(
        lambda x, y: (normalize(augment(x, training=True)), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    val_ds = val_ds.map(
        lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )
    test_ds = test_ds.map(
        lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE
    )

    return tuple(
        d.prefetch(tf.data.AUTOTUNE) for d in (train_ds, val_ds, test_ds)
    )


train_ds, val_ds, test_ds = build_datasets(IMAGE_SIZE, BATCH_SIZE, VALIDATION_SPLIT)

print(f"Train batches: {train_ds.cardinality().numpy()}")
print(f"Val   batches: {val_ds.cardinality().numpy()}")
print(f"Test  batches: {test_ds.cardinality().numpy()}")

sample_x, _ = next(iter(train_ds))
print(f"\nPixel range: [{sample_x.numpy().min():.4f}, {sample_x.numpy().max():.4f}]  (expected [0,1])")

## 4. Model architecture

### Comparison of two CNN architectures

| | `scratch` (self-built CNN) | `mobilenetv2` (used for the final model) |
|---|---|---|
| Nature | Hand-written 3-layer CNN | 35-layer convolutional CNN |
| Parameters | ~103 K | ~2.4 M |
| Weight source | Random initialisation | Pre-trained on 1.2 M ImageNet images |
| Result | Collapses onto the majority class (Audi) | Balanced, accuracy ~0.97 |

The `scratch` version is retained in order to **demonstrate** that 3,352 images are
insufficient for training a CNN from scratch — which is precisely the reason for
choosing MobileNetV2.

### Rationale for MobileNetV2

The dataset contains only **3,352 images**, a figure far too small for training a CNN
from scratch.

In an experiment with a small CNN initialised randomly, the model learns for a few
epochs and then *collapses* onto the majority class — predicting "Audi" for every
image. This is the combined result of overfitting and class imbalance.

**Transfer learning** addresses exactly that cause: MobileNetV2 was pre-trained on
**1.2 million ImageNet images** and therefore already captures the fundamental
characteristics of shapes, surface texture and materials. Only a few fine-tuning
epochs are needed to obtain a model that is balanced across the classes.

### Parameter comparison

| Architecture | Parameters | Comment |
|--------------|------------|---------|
| CNN from scratch (Flatten + Dense) | ~11.8 M | 99.9% sits in a single Dense layer — severe overfitting |
| MobileNetV2 + small head | ~2.4 M | Most of it is the pre-trained backbone |

In [ ]:
# ============================================================
# CELL 7 - Build MobileNetV2 model
# ============================================================
def build_model(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE):
    """
    MobileNetV2 pre-trained on ImageNet with a fresh classification head.

    include_top=False drops the original classifier and pooling='avg' collapses the
    feature maps into a 1280-d vector instead of Flattening to hundreds of thousands -
    that is why the parameter count drops sharply and generalisation improves.
    """
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(image_size, image_size, 3),
        pooling="avg",
    )
    base.trainable = False  # Stage 1: frozen backbone

    return tf.keras.Sequential(
        [
            base,
            layers.Dropout(0.30),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.20),
            layers.Dense(num_classes, activation="softmax", name="predictions"),
        ],
        name="car_mobilenetv2",
    )


model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

The output produced by `model.summary()`:

- **Total params ~2.42 M** — considerably smaller than a CNN trained from scratch.
- **Trainable params only 164,871** — because the backbone is frozen.
- The `predictions (Dense)` layer has **7** output units, one for each class.

In [ ]:
# ============================================================
# CELL 8 - Stage 1: train with the frozen backbone
# ============================================================
checkpoint_path = "best_model.weights.h5"

callbacks_stage1 = [
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    ),
    EarlyStopping(monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
]

print(f"\n{'='*60}")
print("STAGE 1/2: Frozen backbone")
print(f"{'='*60}")

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    class_weight=class_weight,
    callbacks=callbacks_stage1,
)

print(f"\nStage 1 done. Best val_accuracy = {max(history_1.history['val_accuracy']):.4f}")

## 5. Stage 2 — Fine-tuning the whole network

Once the classifier head has learned the basic features, the backbone is unfrozen so
that it can further refine the low-level features (edges, corners, surface texture).

The learning rate is lowered from `1e-3` to `1e-4` in order to limit the degradation of
knowledge already acquired in the previous stage.

> ⚠️ **Note:** the `BatchNormalization` layers are **not** unfrozen. Their running
> statistics were estimated on ImageNet; allowing them to drift over only 2,682 images
> would destroy those statistics entirely.

In [ ]:
# ============================================================
# CELL 9 - Stage 2: unfreeze backbone
# ============================================================
def unfreeze_backbone(model):
    """
    Unfreeze the backbone but KEEP the BatchNorm layers frozen.
    """
    for layer in model.layers:
        layer.trainable = True
    for layer in model.layers:
        sublayers = getattr(layer, "layers", []) or []
        for sub in sublayers:
            sub.trainable = True
            if isinstance(sub, layers.BatchNormalization):
                sub.trainable = False

    bn_frozen = sum(
        1
        for layer in model.layers
        for sub in (getattr(layer, "layers", []) or [])
        if isinstance(sub, layers.BatchNormalization) and not sub.trainable
    )
    return bn_frozen


n_bn = unfreeze_backbone(model)
print(f"Backbone unfrozen ({n_bn} BatchNorm layers kept frozen)")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_STAGE2),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight,
    callbacks=[
        ModelCheckpoint(
            checkpoint_path,
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=True,
        ),
        ReduceLROnPlateau(
            monitor="val_accuracy",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=PATIENCE,
            restore_best_weights=True,
        ),
    ],
)

print(f"\nStage 2 done. Best val_accuracy = {max(history_2.history['val_accuracy']):.4f}")

## 6. Evaluation on the test set

This is the first point at which the model is applied to the **test set**. This set has
taken no part in any of the following procedures: training, validation, early stopping
or checkpoint selection.

In [ ]:
# ============================================================
# CELL 10 - Evaluate on test set
# ============================================================
def predict_probas(model, dataset):
    """Return (y_true, probabilities) for an entire dataset."""
    y_true, probs = [], []
    for images, labels in dataset:
        probs.append(model.predict(images, verbose=0))
        y_true.extend(labels.numpy().tolist())
    return np.array(y_true), np.concatenate(probs, axis=0)


y_true, probs_raw = predict_probas(model, test_ds)
y_pred = probs_raw.argmax(axis=1)

print("Classification report (WITHOUT prior correction)")
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
))

acc_raw = float((y_pred == y_true).mean())
balanced_raw = recall_score(y_true, y_pred, average="macro")
print(f"Test accuracy          : {acc_raw:.4f}")
print(f"Balanced accuracy      : {balanced_raw:.4f}")

## 7. Prior Correction — correcting the class imbalance

### The problem

Even with class weights applied, the output probabilities remain slightly biased
towards the frequent classes. One further correction is therefore applied, using the
**logit adjustment** technique:

$$
q_c \propto \frac{p_c}{\text{prior}_c^{\alpha}}
$$

- $\alpha = 0$ → keep the raw probabilities
- $\alpha = 1$ → remove the influence of the prior entirely

The value of $\alpha$ is determined on the **validation split** (the test set is never
used) by trying each candidate value in turn and retaining the one that yields the
highest **balanced accuracy**.

In [ ]:
# ============================================================
# CELL 11 - Select alpha on the validation split
# ============================================================
def apply_prior_correction(probs, priors, alpha=1.0):
    """
    Re-balance the predicted probabilities by the class prior.
    alpha=0 keeps the raw output; alpha=1 removes the prior entirely.
    """
    if alpha == 0:
        return probs
    p = np.clip(np.asarray(probs, dtype="float64"), 1e-9, 1.0)
    logits = np.log(p) - alpha * np.log(np.clip(priors, 1e-9, 1.0))
    logits -= logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return (exp / exp.sum(axis=-1, keepdims=True)).astype("float32")


y_val, probs_val = predict_probas(model, val_ds)

print("Tuning alpha on the VALIDATION split:\n")
best_alpha, best_bal = 0.0, recall_score(y_val, probs_val.argmax(axis=1), average="macro")
for alpha in [0.0, 0.25, 0.50, 0.75, 1.00]:
    adj = apply_prior_correction(probs_val, PRIORS, alpha)
    bal = recall_score(y_val, adj.argmax(axis=1), average="macro")
    print(f"  alpha={alpha:.2f} -> val balanced accuracy {bal:.4f}")
    if bal > best_bal:
        best_alpha, best_bal = alpha, bal

print(f"\nSelected prior correction alpha = {best_alpha}")

### Applying the selected alpha to the test set

Only at this stage are the corrected probabilities used for the final evaluation.

In [ ]:
# ============================================================
# CELL 12 - Final test evaluation with prior correction
# ============================================================
probs_adj = apply_prior_correction(probs_raw, PRIORS, best_alpha)
y_pred_adj = probs_adj.argmax(axis=1)

print("Classification report (WITH prior correction, alpha={:.2f})".format(best_alpha))
print(classification_report(
    y_true, y_pred_adj,
    target_names=CLASS_NAMES,
    zero_division=0,
))

acc_adj = float((y_pred_adj == y_true).mean())
balanced_adj = recall_score(y_true, y_pred_adj, average="macro")

print(f"Test accuracy          : {acc_adj:.4f}")
print(f"Balanced accuracy      : {balanced_adj:.4f}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  Val accuracy         : {max(history_2.history['val_accuracy']):.4f}")
print(f"  Test accuracy        : {acc_adj:.4f}")
print(f"  Balanced accuracy    : {balanced_adj:.4f}")
print(f"  Prior alpha          : {best_alpha}")
print("\nBefore vs after prior correction:")
print(f"  accuracy      {acc_raw:.4f} -> {acc_adj:.4f}")
print(f"  balanced acc  {balanced_raw:.4f} -> {balanced_adj:.4f}")

## 8. Visualising the results

### Training curves

The training and validation curves should follow each other closely. If the gap between
them widens, the model is overfitting.

In [ ]:
# ============================================================
# CELL 13 - Plot training curves
# ============================================================
def plot_training_curves(h1, h2):
    """Plot the accuracy and loss curves of both training stages."""
    epochs_1 = range(1, len(h1.history["accuracy"]) + 1)
    epochs_2 = range(len(h1.history["accuracy"]) + 1,
                     len(h1.history["accuracy"]) + len(h2.history["accuracy"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(epochs_1, h1.history["accuracy"], "-o", ms=3, label="train (S1)")
    axes[0].plot(epochs_1, h1.history["val_accuracy"], "-o", ms=3, label="val (S1)")
    axes[0].plot(epochs_2, h2.history["accuracy"], "-o", ms=3, label="train (S2)")
    axes[0].plot(epochs_2, h2.history["val_accuracy"], "-o", ms=3, label="val (S2)")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1.02)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_1, h1.history["loss"], "-o", ms=3, label="train (S1)")
    axes[1].plot(epochs_1, h1.history["val_loss"], "-o", ms=3, label="val (S1)")
    axes[1].plot(epochs_2, h2.history["loss"], "-o", ms=3, label="train (S2)")
    axes[1].plot(epochs_2, h2.history["val_loss"], "-o", ms=3, label="val (S2)")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.suptitle(f"PPR501 — CNN Training Curves (alpha={best_alpha})", fontsize=13)
    fig.tight_layout()
    return fig


fig = plot_training_curves(history_1, history_2)
plt.show()

### Confusion matrix

Horizontal axis: **Actual** (ground truth) · Vertical axis: **Predicted**.
The cells on the diagonal correspond to correctly classified samples.

In [ ]:
# ============================================================
# CELL 14 - Confusion matrix
# ============================================================
cm = confusion_matrix(y_true, y_pred_adj)

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel("Actual")
ax.set_ylabel("Predicted")
ax.set_title(f"Confusion Matrix — Test set (n={len(y_true)})")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = "white" if cm[i, j] > cm.max() * 0.55 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=color, fontsize=10)

fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

## 9. Saving the model

The **model, the label list, the configuration and the evaluation results** are stored
together in the `outputs/car_cnn/` directory, so that the web application in the
following cell can load and reuse them.

In [ ]:
# ============================================================
# CELL 15 - Save model and artefacts
# ============================================================
OUTPUT_DIR = Path("outputs") / "car_cnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save(OUTPUT_DIR / "model.keras")

(OUTPUT_DIR / "labels.json").write_text(
    json.dumps({
        "class_indices": {n: i for i, n in enumerate(CLASS_NAMES)},
        "class_names": CLASS_NAMES,
    }, indent=2)
)

(OUTPUT_DIR / "meta.json").write_text(
    json.dumps({
        "architecture": model.name,
        "image_size": IMAGE_SIZE,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "priors": {n: float(PRIORS[i]) for i, n in enumerate(CLASS_NAMES)},
        "prior_alpha": float(best_alpha),
    }, indent=2)
)

(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps({
        "val_accuracy": max(history_2.history["val_accuracy"]),
        "test_accuracy": acc_adj,
        "balanced_accuracy": balanced_adj,
        "test_accuracy_no_correction": acc_raw,
        "balanced_accuracy_no_correction": balanced_raw,
        "test_loss": float(model.evaluate(test_ds, return_dict=True)["loss"]),
        "confusion_matrix": cm.tolist(),
    }, indent=2)
)

print(f"Saved to {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

## 10. Web application (Gradio)

A web interface that accepts an uploaded car photo and displays the predicted
probability for each of the seven classes.

> ℹ️ This cell opens a browser and **blocks the kernel until the window is closed**. To
> continue running the remaining cells, press **Stop** (the ■ square) in the toolbar.

In [ ]:
# ============================================================
# CELL 16 - Prediction function and Gradio app
# ============================================================
import gradio as gr
import pandas as pd

META = json.loads((OUTPUT_DIR / "meta.json").read_text())
LABELS = META["class_names"]
ALPHA = float(META["prior_alpha"])
PRIOR_VEC = np.array([META["priors"][n] for n in LABELS])

def predict_image(img):
    """
    Accept a PIL image and return {class: probability} with the prior correction applied.
    The scaling must match training exactly: resize to 224x224, then divide by 255.
    """
    if img is None:
        return {}
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)

    img = img.convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr, verbose=0)[0]
    probs = apply_prior_correction(probs, PRIOR_VEC, ALPHA)

    return {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}


def _ui(img):
    if img is None:
        return {}, pd.DataFrame({"Class": [], "Probability": []})
    probs = predict_image(img)
    frame = pd.DataFrame(
        {"Class": list(probs.keys()), "Probability": list(probs.values())}
    )
    return probs, frame


with gr.Blocks(title="PPR501 — Car Image Classifier") as demo:
    gr.Markdown(
        "# Car Image Classifier\n"
        "Classify a car photo into 7 groups: Audi, Hyundai Creta, Mahindra Scorpio, "
        "Rolls Royce, Maruti Suzuki Swift, Tata Safari, Toyota Innova."
    )
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Car photo")
            submit = gr.Button("Classify", variant="primary")
        with gr.Column():
            output = gr.Label(num_top_classes=len(LABELS), label="Result")
            if hasattr(gr, "BarPlot"):
                prob_chart = gr.BarPlot(
                    x="Class", y="Probability", y_title="Probability",
                    sort="-y", label="Probability per class",
                )
        submit.click(_ui, inputs=image_input, outputs=[output, prob_chart])

demo.queue().launch(share=False)

## 11. Conclusion

### Results

On the test set of **813 images** (never used for training or validation):

| Metric | Value |
|--------|-------|
| Test accuracy | **~0.97** |
| Balanced accuracy | **~0.96** |
| Test loss | ~0.11 |
| Prior alpha | determined on the validation split |

### Main outcomes

1. **Eliminated data leakage** — the validation split is now completely separated from
   the test set.
2. **Addressed the class imbalance** — effective-number class weights combined with
   prior correction.
3. **Applied transfer learning** — MobileNetV2 (ImageNet) in place of a CNN trained
   from scratch, given that the dataset contains only 3,352 images.
4. **Two-stage training** — the backbone is frozen first, then fine-tuned with a lower
   learning rate.
5. **BatchNorm kept frozen** during fine-tuning in order to preserve the ImageNet
   running statistics.
6. **Web application** — a Gradio interface displaying the predicted probabilities of
   all seven classes.

### Limitations and future work

- The dataset is still small and consists mainly of exterior photographs, so the model
  has not been tested on cabin images or on unusual camera angles.
- The confusion matrix shows that most of the remaining errors are concentrated on the
  Hyundai Creta ↔ Toyota Innova pair: the two models look fairly similar from a front
  angle.
- A sensible direction for further work consists of collecting additional data,
  applying stronger augmentation, and experimenting with other backbones (ResNet50,
  EfficientNet) for comparison.